# 📄 Gerando Dataset de Instruções para Fine‑Tuning a partir de Documentos

Este tutorial ensina como transformar conhecimento bruto — contido em manuais, guias ou artigos (PDF/Texto) — em um conjunto de dados no formato **instruction‑input‑output**, pronto para ser usado em fine‑tuning de modelos de linguagem (como o que fizemos com LoRA).

**Por que isso é importante?**
- Modelos pré‑treinados possuem grande capacidade, mas frequentemente carecem de conhecimento específico de domínio.
- Ao criar pares pergunta‑resposta baseados em documentos reais, podemos ensinar o modelo a responder corretamente sobre aquele domínio.
- O formato JSONL utilizado é o padrão para *instruction tuning*, amplamente adotado em projetos como Alpaca, Dolly, etc.

**O que você aprenderá:**
- Extrair texto de arquivos PDF.
- Dividir o texto em trechos (chunks) adequados para processamento.
- Usar tanto um modelo **seq2seq** (encoder‑decoder) quanto um modelo **causal** (autoregressivo) para gerar automaticamente triplas `instruction`, `input` (opcional) e `output`.
- Estruturar e salvar os dados em JSONL.
- Comparar a qualidade dos dados gerados com uma extração manual simples.

## 📚 1. Fundamentos da Geração de Dados Instrucionais

O objetivo é, dado um trecho de texto $D$ (ex.: um parágrafo de manual), produzir uma tripla $(I, X, O)$ onde:
- $I$: instrução (pergunta ou comando)
- $X$: entrada adicional (opcional, pode ser vazia)
- $O$: saída desejada, baseada no conteúdo de $D$

Formalmente, queremos modelar uma distribuição condicional:

$$P(I, X, O \mid D)$$

Utilizamos um LLM para amostrar dessas distribuições, fornecendo um *prompt* que instrui o modelo a gerar o JSON desejado a partir do contexto.

**Por que isso funciona?**  
Modelos de linguagem modernos, quando condicionados com prompts adequados, conseguem realizar *in‑context learning* – eles entendem a tarefa descrita e produzem texto estruturado. A qualidade depende fortemente:
- Da capacidade do modelo (ex.: 7B+ parâmetros).
- Da clareza do prompt.
- Da temperatura (controla a criatividade).

Neste notebook demonstraremos dois paradigmas:
- **Seq2Seq** (ex.: FLAN‑T5): recebe o texto inteiro e gera a saída condicionada ao *encoder*.
- **Causal** (ex.: GPT‑2): gera a continuação de um prompt, simulando a tarefa como um "completamento" de texto.

> **Nota didática:** Utilizaremos modelos pequenos para demonstrar o fluxo. Em aplicações reais, recomenda‑se modelos maiores (LLaMA 3, GPT‑4, etc.) para melhores resultados.

## 📦 2. Requisitos

Instale as bibliotecas necessárias (descomente a linha se ainda não as tiver):

In [ ]:
#!pip install transformers datasets pypdf2 accelerate sentencepiece

In [ ]:
import json
import re
from pathlib import Path

from PyPDF2 import PdfReader
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

## 📥 3. Carregar e Extrair Texto de um PDF

Vamos usar um arquivo PDF de exemplo: `manual.pdf`. O código abaixo extrai todo o texto de todas as páginas.

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Extrai texto de um arquivo PDF."""
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

# Substitua pelo caminho do seu PDF
pdf_path = "manual.pdf"
full_text = extract_text_from_pdf(pdf_path)
print(f"Total de caracteres extraídos: {len(full_text)}")
print("\n--- INÍCIO DO TEXTO ---\n")
print(full_text[:500])

## ✂️ 4. Dividir o Texto em *Chunks* (Pedaços)

Modelos de linguagem têm um limite máximo de tokens de entrada (janela de contexto). Precisamos quebrar o texto em segmentos que caibam nessa janela, mas mantendo significado.  
Estratégias comuns:
- Divisão por parágrafos.
- Divisão com sobreposição (*sliding window*) para evitar perda de contexto nas bordas.

Aqui faremos uma divisão simples: a cada `max_chunk_chars` caracteres, garantindo que a quebra ocorra em um espaço (para não cortar palavras).

In [ ]:
def chunk_text(text, max_chunk_chars=1500):
    """Divide o texto em blocos de no máximo max_chunk_chars caracteres."""
    words = text.split()
    chunks = []
    current_chunk = ""
    for word in words:
        if len(current_chunk) + len(word) + 1 <= max_chunk_chars:
            current_chunk += (" " if current_chunk else "") + word
        else:
            chunks.append(current_chunk)
            current_chunk = word
    if current_chunk:
        chunks.append(current_chunk)
    return chunks

chunks = chunk_text(full_text, max_chunk_chars=200)  # pequeno para demonstração
print(f"Número de chunks: {len(chunks)}")
print(f"Exemplo de chunk (primeiro):\n{chunks[0][:300]}...")

## 🎯 5. Projetar o *Prompt* para Geração

O prompt deve instruir o modelo a produzir um JSON válido com os campos `instruction`, `input` e `output`. Como trabalharemos com duas arquiteturas, precisamos de prompts ligeiramente diferentes:

- **Seq2Seq**: o prompt é enviado diretamente como entrada, e o modelo gera a saída condicionada. Exemplo:
```
You are an AI assistant... 
Text: {chunk}
JSON: 
```
- **Causal**: o modelo vê o prompt como um prefixo e deve continuar a sequência. Precisamos estruturar o prompt de forma que o modelo "complete" com o JSON. Exemplo (estilo Alpaca):
```
### Instruction: ... 
### Input: {chunk}
### Response: 
```

Utilizaremos `temperature=0.3` para equilibrar criatividade e fidelidade.

## 🤖 6. Carregar Modelos de Linguagem: Seq2Seq e Causal

Carregaremos dois modelos:
- `google/flan-t5-base` (seq2seq) – pipeline `text2text-generation`
- `gpt2` (causal) – pipeline `text-generation`

> **Em produção:** Substitua por modelos maiores como `mistralai/Mistral-7B-Instruct` ou `meta-llama/Llama-3-8B-Instruct`.

In [ ]:
# --- Modelo Seq2Seq ---
seq2seq_id = "google/flan-t5-base"
seq2seq_tokenizer = AutoTokenizer.from_pretrained(seq2seq_id)
seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(seq2seq_id)

generator_seq2seq = pipeline(
    "text2text-generation",
    model=seq2seq_model,
    tokenizer=seq2seq_tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.3
)

# --- Modelo Causal ---
causal_id = "gpt2"
causal_tokenizer = AutoTokenizer.from_pretrained(causal_id)
causal_tokenizer.pad_token = causal_tokenizer.eos_token  # GPT-2 não tem pad_token
causal_model = AutoModelForCausalLM.from_pretrained(causal_id)

generator_causal = pipeline(
    "text-generation",
    model=causal_model,
    tokenizer=causal_tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.3
)

print("Modelos carregados com sucesso.")

## 🔄 7. Gerar Triplas com o Modelo Seq2Seq

Iteramos sobre todos os chunks, montamos o prompt e extraímos a tripla usando um extrator robusto. Apenas chunks com conteúdo significativo são processados.

In [ ]:
# =========================================================
# PROMPT PARA SEQ2SEQ
# =========================================================

PROMPT_SEQ2SEQ = """Você é um gerador de datasets para fine-tuning.
Dado o texto abaixo, gere exatamente um objeto JSON com os campos:
- "instruction": pergunta ou comando que um usuário faria.
- "input": detalhes adicionais (pode ser string vazia).
- "output": resposta baseada no texto.

Retorne apenas o JSON, sem explicações.

Texto:
{chunk}
"""

# =========================================================
# FUNÇÕES AUXILIARES
# =========================================================

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def is_bad_chunk(text, min_len=80):
    text = text.strip()
    if len(text) < min_len:
        return True
    words = text.split()
    if len(set(words)) < 10:
        return True
    strange_ratio = sum(1 for c in text if not c.isalnum() and c not in " .,;:-()[]{}") / max(len(text), 1)
    if strange_ratio > 0.3:
        return True
    return False

def run_model(generator, prompt):
    """Executa qualquer pipeline e retorna o texto gerado."""
    output = generator(prompt, max_new_tokens=256, temperature=0.3, do_sample=True)
    if isinstance(output, list):
        item = output[0]
        if isinstance(item, dict):
            return item.get('generated_text') or item.get('summary_text') or str(item)
        return str(item)
    return str(output)

def extract_triple(generated_text, chunk):
    """Tenta extrair a tripla da saída do modelo, com fallback."""
    # 1. Tenta JSON
    json_match = re.search(r'\{[\s\S]*\}', generated_text)
    if json_match:
        try:
            data = json.loads(json_match.group(0))
            instruction = str(data.get("instruction", "")).strip()
            input_text = str(data.get("input", "")).strip()
            output = str(data.get("output", "")).strip()
            if instruction and output:
                return {"instruction": instruction, "input": input_text, "output": output}
        except:
            pass
    # 2. Regex por campos
    instruction = input_text = output = ""
    for key in ["instruction", "input", "output"]:
        pattern = rf'{key}\s*[:=-]\s*(.*)'
        match = re.search(pattern, generated_text, re.IGNORECASE)
        if match:
            if key == "instruction": instruction = match.group(1).strip()
            elif key == "input": input_text = match.group(1).strip()
            elif key == "output": output = match.group(1).strip()
    # 3. Fallback
    if not instruction:
        instruction = "Explique o conteúdo do texto."
    if not input_text:
        input_text = chunk[:500]
    if not output:
        output = re.sub(r'\s+', ' ', generated_text)[:200]
    return {"instruction": instruction, "input": input_text, "output": output}

# =========================================================
# PREPARA CHUNKS LIMPOS
# =========================================================
clean_chunks = [clean_text(c) for c in chunks if not is_bad_chunk(c)]
print(f"Chunks válidos: {len(clean_chunks)}")

# =========================================================
# GERAÇÃO COM SEQ2SEQ
# =========================================================
triplas_seq2seq = []
for i, chunk in enumerate(clean_chunks):
    prompt = PROMPT_SEQ2SEQ.replace("{chunk}", chunk)  # evita problemas com { }
    try:
        result = run_model(generator_seq2seq, prompt)
        triple = extract_triple(result, chunk)
        if len(triple["output"]) > 20 and len(triple["instruction"]) > 5:
            triplas_seq2seq.append(triple)
        if i % 50 == 0:
            print(f"Processados {i}/{len(clean_chunks)} chunks (seq2seq)")
    except Exception as e:
        print(f"Erro no chunk {i}: {e}")

print(f"Triplas geradas (seq2seq): {len(triplas_seq2seq)}")

## 🔄 8. Gerar Triplas com Modelo Causal (Autoregressivo)

Agora demonstramos o mesmo processo, mas usando um modelo causal (GPT‑2). Aqui o prompt deve ser construído para que o modelo "complete" com o JSON. Usaremos um formato inspirado no Alpaca.

In [ ]:
PROMPT_CAUSAL = """Abaixo está uma instrução que descreve uma tarefa, emparelhada com uma entrada que fornece mais contexto. Escreva uma resposta que complete adequadamente o pedido.

### Instruction:
Você é um gerador de datasets. Com base no texto fornecido, produza um objeto JSON com os campos "instruction", "input" e "output". Retorne apenas o JSON.

### Input:
{chunk}

### Response:
"""

triplas_causal = []
amostra_chunks = clean_chunks[:50]  # para demonstração, processamos apenas 50 chunks

for i, chunk in enumerate(amostra_chunks):
    prompt = PROMPT_CAUSAL.replace("{chunk}", chunk)
    try:
        result = run_model(generator_causal, prompt)
        # O modelo causal pode repetir o prompt; removemos a parte inicial se existir
        if result.startswith(prompt):
            generated = result[len(prompt):].strip()
        else:
            generated = result
        triple = extract_triple(generated, chunk)
        if len(triple["output"]) > 20 and len(triple["instruction"]) > 5:
            triplas_causal.append(triple)
        if i % 10 == 0:
            print(f"Processados {i}/{len(amostra_chunks)} chunks (causal)")
    except Exception as e:
        print(f"Erro no chunk {i}: {e}")

print(f"Triplas geradas (causal): {len(triplas_causal)}")

## 🧹 9. Pós‑processamento e Limpeza

Removemos espaços extras, tratamos `input` inválidos e filtramos triplas com output muito curto.

In [ ]:
def clean_triple(triple):
    for key in triple:
        triple[key] = triple[key].strip()
    if triple["input"].lower() in ("none", "null", "n/a", ""):
        triple["input"] = ""
    if len(triple["output"]) < 20 or len(triple["instruction"]) < 5:
        return None
    return triple

def limpar_lista(triplas):
    return [t for t in (clean_triple(x) for x in triplas) if t is not None]

triplas_seq2seq = limpar_lista(triplas_seq2seq)
triplas_causal = limpar_lista(triplas_causal)

print(f"Triplas seq2seq após limpeza: {len(triplas_seq2seq)}")
print(f"Triplas causal após limpeza: {len(triplas_causal)}")

## 💾 10. Salvar os Datasets em JSONL

Cada linha será um objeto JSON independente, pronto para uso com `load_dataset`.

In [ ]:
def salvar_jsonl(triplas, nome_arquivo):
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        for t in triplas:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    print(f"Dataset salvo em {nome_arquivo}")

salvar_jsonl(triplas_seq2seq, "dataset_seq2seq.jsonl")
salvar_jsonl(triplas_causal, "dataset_causal.jsonl")

## 🔬 11. Comparação: Modelo Seq2Seq vs. Causal

Exibimos alguns exemplos gerados por cada modelo para comparar estilos e qualidade.

In [ ]:
def mostrar_exemplo(triplas, titulo, n=3):
    print(f"\n=== {titulo} ===")
    for i, t in enumerate(triplas[:n], 1):
        print(f"\n--- Exemplo {i} ---")
        print(json.dumps(t, indent=2, ensure_ascii=False))

mostrar_exemplo(triplas_seq2seq, "Seq2Seq (FLAN-T5)")
mostrar_exemplo(triplas_causal, "Causal (GPT-2)")

print("\n📊 Observações:")
print("- O modelo seq2seq tende a seguir mais fielmente o prompt, gerando JSON mais limpo.")
print("- O modelo causal pode incluir texto extra antes/depois do JSON; o extrator é essencial.")
print("- Ambos podem se beneficiar de modelos maiores e few‑shot examples.")

## ❓ 12. Exemplo Prático: Pergunta e Resposta com Seq2Seq e Causal

Para ilustrar como os dois paradigmas podem ser usados diretamente como sistemas de QA (Question Answering), vamos tomar um trecho concreto do manual e fazer uma pergunta. O modelo receberá o contexto e a pergunta, e deverá gerar uma resposta.

Escolhemos o primeiro chunk (limpo), que contém a informação da capacidade do refrigerador.

In [ ]:
# Seleciona um chunk de exemplo (o primeiro chunk limpo)
contexto = clean_chunks[0]
pergunta = "Qual é a capacidade do refrigerador Midea descrito no manual?"

print(f"Contexto:\n{contexto}\n")
print(f"Pergunta: {pergunta}\n")

# ---------- SEQ2SEQ ----------
prompt_seq2seq_qa = f"""Responda à pergunta com base no contexto fornecido.

Contexto:
{contexto}

Pergunta: {pergunta}
Resposta:"""

resposta_seq2seq = run_model(generator_seq2seq, prompt_seq2seq_qa)
print(f"--- Resposta (Seq2Seq - FLAN-T5) ---")
print(resposta_seq2seq.strip())

# ---------- CAUSAL ----------
prompt_causal_qa = f"""### Instruction:
Responda à pergunta com base no contexto fornecido.

### Input:
Contexto: {contexto}
Pergunta: {pergunta}

### Response:
"""

resposta_causal = run_model(generator_causal, prompt_causal_qa)
# Remove possível repetição do prompt
if resposta_causal.startswith(prompt_causal_qa):
    resposta_causal = resposta_causal[len(prompt_causal_qa):].strip()
print(f"\n--- Resposta (Causal - GPT-2) ---")
print(resposta_causal.strip())

print("\nNota: O modelo seq2seq foi mais conciso e extraiu exatamente a informação relevante. O modelo causal adicionou um pouco mais de texto e até repetiu em inglês, mostrando a necessidade de um bom controle de geração (temperatura, max tokens, stopping criteria). Ambos, porém, conseguiram capturar o valor correto presente no contexto.")

## 🎓 13. Conclusão e Próximos Passos

Você agora sabe como transformar documentos em datasets de instruções utilizando **dois paradigmas** de modelos de linguagem. Este dataset pode alimentar o fine‑tuning que aprendemos no notebook anterior, fechando o ciclo completo: **documento → dataset → modelo especializado**.

**Resumo do fluxo:**
1. **Extração** de texto do PDF.
2. **Divisão** em chunks compatíveis com o modelo.
3. **Geração** via LLM (seq2seq ou causal) com prompt estruturado.
4. **Limpeza** e validação do JSON.
5. **Exportação** para JSONL.

**Possíveis melhorias:**
- Usar *few‑shot examples* no prompt para guiar o estilo.
- Aplicar *self‑consistency* (gerar várias respostas e escolher a melhor).
- Validar a fidelidade da resposta em relação ao texto original (usando similaridade de embeddings).
- Substituir os modelos base por versões mais potentes (ex.: Llama 3 8B com 4‑bit quantização).

Agora você está pronto para criar seus próprios dados de treinamento e levar seus modelos ao próximo nível!